# 特征筛选

1. 在第一个代码单元设置样本表、变量表、Y 标签、OOT 切分等参数；
2. 第二个单元列出样本表里可选的 Y 标签；
3. 运行第三个单元开始筛选。

OOT 只按 `sample_datetime` 时间切分，默认取最近 30%，切点落在时间点边界上；时间列只有一个取值时会报错。切点只用训练期拟合，OOT 只用于评估。入选需同时满足 IV 阈值、单调、OOT 稳定（方向与 PSI）、缺失箱 IV 占比不超限。结果写入数据根目录下的 `feature_screening_reports/`，文件名带标签名。

In [ ]:
from pathlib import Path

import importlib
import feature_screening as fs

importlib.reload(fs)  # 修改 feature_screening.py 后重跑本单元即可生效

ROOT_DIR = Path(fs.DEFAULT_ROOT_DIR)  # 固定数据根目录，与 notebook 位置无关
SAMPLE_FILE = "samples.csv"
VARIABLE_FILE = "variable_bank_statement_surplus.csv"
TARGET = "y_fpd15_ovd"  # 可选 y_fpd15_ovd / y_fpd7_ovd / y_mob1_5plus 等
OBS_COL = None          # None 时按 TARGET 自动匹配对应 *_obs
FEATURE_PREFIXES = None  # None 表示变量表除主键外全部列；也可传元组限定前缀
MIN_IV = 0.02
OOT_START = None            # 指定 OOT 起始日期（如 "2026-10-01"），优先于 OOT_RATIO
OOT_RATIO = 0.3             # 按 sample_datetime 取最近 30% 作 OOT；None 关闭 OOT 评估
MAX_PSI = 0.25              # OOT 与训练集分箱占比的最大 PSI
MAX_MISSING_IV_SHARE = 0.5  # 缺失箱 IV 占整体 IV 的上限；None 不限制

In [ ]:
import pandas as pd

sample_columns = pd.read_csv(ROOT_DIR / SAMPLE_FILE, nrows=0).columns
print("样本表可选 Y 标签：")
print([c for c in sample_columns if c.startswith("y_")])
print("变量表列数：", len(pd.read_csv(ROOT_DIR / VARIABLE_FILE, nrows=0).columns))

In [ ]:
data, summary, bin_detail, candidates = fs.run_feature_screening(
    ROOT_DIR,
    min_iv=MIN_IV,
    sample_file=SAMPLE_FILE,
    variable_file=VARIABLE_FILE,
    target=TARGET,
    obs_col=OBS_COL,
    feature_prefixes=FEATURE_PREFIXES,
    oot_start=OOT_START,
    oot_ratio=OOT_RATIO,
    max_psi=MAX_PSI,
    max_missing_iv_share=MAX_MISSING_IV_SHARE,
)

summary.head(30)